# 02 · Unificación de las 9 tablas  *(sección 1 del script)*

**Qué hacemos:** unimos las 9 tablas crudas de Kaggle en un único dataset a nivel de **"ítem de pedido"** (cada fila = un producto dentro de un pedido), agregando pagos y reviews a nivel de pedido *antes* de unirlos para no duplicar filas, y cruzando la geolocalización tanto del **cliente** como del **vendedor**.

**Para qué:** porque el dataset original replica una base de datos operativa real, **normalizada en 9 tablas relacionadas**: hasta que no se integran en una sola tabla, no se puede analizar el negocio "por pedido", "por producto" o "por vendedor" de forma conjunta.

**Salida de este notebook:** `data/olist_dataset_unificado.csv` (lo consume el notebook 03).

## Las 9 tablas y su granularidad

| Tabla | 1 fila equivale a… | Clave principal / foránea |
|---|---|---|
| `olist_orders_dataset` | 1 **pedido** | `order_id` |
| `olist_order_items_dataset` | 1 **producto dentro de un pedido** (`order_id` + `order_item_id`) | trae `product_id` y `seller_id` |
| `olist_order_payments_dataset` | 1 **transacción de pago** (un pedido puede tener varias) | `order_id` |
| `olist_order_reviews_dataset` | 1 **reseña** (un pedido puede tener varias) | `order_id` |
| `olist_customers_dataset` | 1 **cliente** | `customer_id` (va en orders) |
| `olist_products_dataset` | 1 **producto** | `product_id` (va en items) |
| `olist_sellers_dataset` | 1 **vendedor** | `seller_id` (va en items) |
| `olist_geolocation_dataset` | 1 **coordenada** (varias por código postal) | `geolocation_zip_code_prefix` |
| `product_category_name_translation` | 1 **categoría PT → EN** | `product_category_name` |

### La regla de oro de este cruce

> El **lado izquierdo** del `merge` define la granularidad del resultado.

Partimos de `items` (que ya tiene una fila por producto-por-pedido) y hacemos `how="left"` hacia todas las tablas "uno a uno" (orders, products, sellers, customers): así **la cantidad de filas no cambia**. Las tablas que traen *muchas* filas por pedido — `payments` y `reviews` — primero las **agregamos a un renglón por pedido**; si las unimos crudas, cada pedido se duplicaría tantas veces como pagos/reseñas tenga y los conteos posteriores serían falsos.

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 50)

# Raíz del proyecto: si el kernel corre desde notebooks/, subimos un nivel
BASE = Path.cwd()
if not (BASE / "data").exists() and (BASE.parent / "data").exists():
    BASE = BASE.parent

RAW_DIR = BASE / "data" / "raw"
CSV_UNIFICADO = BASE / "data" / "olist_dataset_unificado.csv"

orders = pd.read_csv(RAW_DIR / "olist_orders_dataset.csv")
items = pd.read_csv(RAW_DIR / "olist_order_items_dataset.csv")
payments = pd.read_csv(RAW_DIR / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(RAW_DIR / "olist_order_reviews_dataset.csv")
customers = pd.read_csv(RAW_DIR / "olist_customers_dataset.csv")
products = pd.read_csv(RAW_DIR / "olist_products_dataset.csv")
sellers = pd.read_csv(RAW_DIR / "olist_sellers_dataset.csv")
geolocation = pd.read_csv(RAW_DIR / "olist_geolocation_dataset.csv")
cat_translation = pd.read_csv(RAW_DIR / "product_category_name_translation.csv")

tabla_shapes = pd.DataFrame(
    [(nombre, t.shape[0], t.shape[1]) for nombre, t in [
        ("orders (pedidos)", orders),
        ("items (productos por pedido)", items),
        ("payments (pagos)", payments),
        ("reviews (reseñas)", reviews),
        ("customers (clientes)", customers),
        ("products (productos)", products),
        ("sellers (vendedores)", sellers),
        ("geolocation (coordenadas)", geolocation),
        ("category translation (categorías EN)", cat_translation),
    ]],
    columns=["tabla", "filas", "columnas"],
)
tabla_shapes

,tabla,filas,columnas
0,orders (pedidos),99441,8
1,items (productos por pedido),112650,7
2,payments (pagos),103886,5
3,reviews (reseñas),99224,7
4,customers (clientes),99441,5
5,products (productos),32951,9
6,sellers (vendedores),3095,4
7,geolocation (coordenadas),1000163,5
8,category translation (categorías EN),71,2


**Insight:** se ven claramente las dos tablas "muchos a muchos respecto del pedido": `items` (~112 mil filas) tiene **más filas que `orders`** (~100 mil pedidos) porque un pedido puede traer varios productos, y `geolocation` es la más grande porque trae muchas coordenadas por código postal. `payments` y `reviews` también tienen filas "por transacción/reseña", no "por pedido" — por eso el próximo paso las agrega antes de cruzarlas.

## Pagos agregados a nivel pedido

**Qué hacemos:** agrupamos `payments` por `order_id` y calculamos:

- `payment_value_total`: **suma** de todo lo pagado en el pedido (un pedido pagado con 2 tarjetas se suma).
- `payment_installments_max`: **máximo** de cuotas elegidas (la cuota más alta entre los pagos).
- `payment_type_principal`: medio de pago **más frecuente** en el pedido (`value_counts().idxmax()`).

**Para qué:** para que al unir, cada pedido aporte **una sola fila** de información de pago, sin duplicar ítems.

In [2]:
payments_agg = (
    payments.groupby("order_id")
    .agg(
        payment_value_total=("payment_value", "sum"),
        payment_installments_max=("payment_installments", "max"),
        payment_type_principal=(
            "payment_type",
            lambda x: x.value_counts().idxmax() if x.notna().any() else None,
        ),
    )
    .reset_index()
)

pagos_por_pedido = payments.groupby("order_id").size()
print(f"Pagos por pedido: mínimo {pagos_por_pedido.min()}, máximo {pagos_por_pedido.max()}, "
      f"promedio {pagos_por_pedido.mean():.2f}")
print(f"Pedidos agregados: {len(payments_agg):,}")
payments_agg.head()

Pagos por pedido: mínimo 1, máximo 29, promedio 1.04
Pedidos agregados: 99,440


,order_id,payment_value_total,payment_installments_max,payment_type_principal
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,2,credit_card
1,00018f77f2f0320c557190d7a144bdd3,259.83,3,credit_card
2,000229ec398224ef6ca0657da4fc703e,216.87,5,credit_card
3,00024acbcdf0a6daa1e931b038114c75,25.78,2,credit_card
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,3,credit_card


**Insight:** aunque la mayoría de los pedidos se paga en una sola transacción, hay pedidos con más de un pago — si no los hubiéramos agregado, esos pedidos aparecerían repetidos y el promedio de "precio" quedaría multiplicado por la cantidad de pagos.

## Reseñas agregadas a nivel pedido

**Qué hacemos:** ordenamos las reviews por fecha de creación y, si un pedido tiene más de una, **nos quedamos con la última** (`.groupby("order_id").last()`), conservando `review_score` y `review_comment_message`.

**Para qué:** misma lógica que con pagos: un renglón por pedido. Además, quedarse con la **última** reseña refleja mejor la opinión final del cliente (la más cercana a que el pedido terminara).

In [3]:
reviews_agg = (
    reviews.sort_values("review_creation_date")
    .groupby("order_id")
    .last()
    .reset_index()[["order_id", "review_score", "review_comment_message"]]
)

print(f"Reviews crudas: {len(reviews):,} filas -> {len(reviews_agg):,} pedidos con reseña")
reviews_agg.head()

Reviews crudas: 99,224 filas -> 98,673 pedidos con reseña


,order_id,review_score,review_comment_message
0,00010242fe8c5a6d1ba2dd792cb16214,5,"Perfeito, produto entregue antes do combinado."
1,00018f77f2f0320c557190d7a144bdd3,4,None
2,000229ec398224ef6ca0657da4fc703e,5,Chegou antes do prazo previsto e o produto sur...
3,00024acbcdf0a6daa1e931b038114c75,4,None
4,00042b26cf59d7ce69dfabb4e55b4fd9,5,Gostei pois veio no prazo determinado .


## Categorías de producto en inglés

**Qué hacemos:** al dataset de `products` le unimos la tabla de traducción (`product_category_name_translation`) por `product_category_name`, con `how="left"`: a cada producto le agregamos su nombre de categoría en inglés.

**Para qué:** el análisis se reporta en inglés/español y la columna original está en portugués; además, el `left` deja en `NaN` las categorías sin traducir, que el notebook 03 rellenará con `"sin_categoria"`.

In [4]:
sin_categoria_antes = products["product_category_name"].isna().sum()

products = products.merge(cat_translation, on="product_category_name", how="left")

sin_traducir = products["product_category_name_english"].isna().sum()
print(f"Productos: {len(products):,}")
print(f"Con categoría vacía de origen: {sin_categoria_antes}")
print(f"Quedaron sin traducir (en inglés NaN): {sin_traducir}")

Productos: 32,951
Con categoría vacía de origen: 610
Quedaron sin traducir (en inglés NaN): 623


## Cadena de merges (ítems + pedidos + productos + vendedores + clientes + pagos + reseñas)

**Qué hacemos:** encadenamos los cruces con `how="left"`, imprimiendo la forma (filas × columnas) **después de cada merge** para ver que la granularidad no cambie.

**Para qué:** el left-join desde `items` garantiza que el resultado siga teniendo **una fila por producto dentro de pedido**; solo se agregan columnas, nunca filas.

In [5]:
df = items.merge(orders, on="order_id", how="left")
print(f"items + orders    -> {df.shape[0]:,} filas x {df.shape[1]} columnas (granularidad intacta)")

df = df.merge(products, on="product_id", how="left")
print(f"+ products        -> {df.shape[0]:,} filas x {df.shape[1]} columnas")

df = df.merge(sellers, on="seller_id", how="left")
print(f"+ sellers         -> {df.shape[0]:,} filas x {df.shape[1]} columnas")

df = df.merge(customers, on="customer_id", how="left")
print(f"+ customers       -> {df.shape[0]:,} filas x {df.shape[1]} columnas")

df = df.merge(payments_agg, on="order_id", how="left")
print(f"+ payments_agg    -> {df.shape[0]:,} filas x {df.shape[1]} columnas (agregado: no duplica)")

df = df.merge(reviews_agg, on="order_id", how="left")
print(f"+ reviews_agg     -> {df.shape[0]:,} filas x {df.shape[1]} columnas (agregado: no duplica)")

print()
unico = not df.duplicated(["order_id", "order_item_id"]).any()
print(f"Grano final: {df.shape[0]:,} filas x {df.shape[1]} columnas")
print(f"order_id + order_item_id es único: {unico}")

items + orders    -> 112,650 filas x 14 columnas (granularidad intacta)


+ products        -> 112,650 filas x 23 columnas


+ sellers         -> 112,650 filas x 26 columnas


+ customers       -> 112,650 filas x 30 columnas


+ payments_agg    -> 112,650 filas x 33 columnas (agregado: no duplica)


+ reviews_agg     -> 112,650 filas x 35 columnas (agregado: no duplica)

Grano final: 112,650 filas x 35 columnas
order_id + order_item_id es único: True


**Insight:** la cantidad de filas **no cambió en ningún merge** — es la comprobación de que la unificación respetó el grano de "ítem de pedido" y de que pagos/reseñas se agregaron correctamente antes del cruce. Que `order_id + order_item_id` sea único confirma que cada fila es efectivamente un ítem distinto de un pedido distinto.

## Geolocalización del cliente y del vendedor

**Qué hacemos:**

1. Promediamos `lat`/`lng` por `geolocation_zip_code_prefix` (la tabla trae varias coordenadas por código postal → un promedio por zip).
2. Usamos **esa misma tabla dos veces**: renombrando las columnas para unirla como geolocalización del **cliente** (`customer_lat/lng`, por `customer_zip_code_prefix`) y del **vendedor** (`seller_lat/lng`, por `seller_zip_code_prefix`).

**Para qué:** las coordenadas del cliente sirven para los mapas (notebook 08); las del vendedor son imprescindibles para calcular la **distancia real cliente-vendedor** (feature `distancia_km`, notebook 04), que es el corazón del análisis logístico.

In [6]:
geo_prom = (
    geolocation.groupby("geolocation_zip_code_prefix")
    .agg(lat=("geolocation_lat", "mean"), lng=("geolocation_lng", "mean"))
    .reset_index()
)

geo_cliente = geo_prom.rename(columns={
    "geolocation_zip_code_prefix": "customer_zip_code_prefix",
    "lat": "customer_lat", "lng": "customer_lng",
})
df = df.merge(geo_cliente, on="customer_zip_code_prefix", how="left")

geo_vendedor = geo_prom.rename(columns={
    "geolocation_zip_code_prefix": "seller_zip_code_prefix",
    "lat": "seller_lat", "lng": "seller_lng",
})
df = df.merge(geo_vendedor, on="seller_zip_code_prefix", how="left")

print(f"Filas tras geolocalización: {df.shape[0]:,} (no cambió)")
print(f"Sin coordenadas de cliente : {df['customer_lat'].isna().sum():,} filas")
print(f"Sin coordenadas de vendedor: {df['seller_lat'].isna().sum():,} filas")

Filas tras geolocalización: 112,650 (no cambió)
Sin coordenadas de cliente : 302 filas
Sin coordenadas de vendedor: 253 filas


**Insight:** la granularidad sigue intacta y los nulos de coordenadas que queden corresponden a códigos postales que no matchean contra la tabla de geolocalización (o a zonas fuera del rango de Brasil, que el notebook 03 se encarga de poner en `NaN` de forma explícita). No se borra ninguna fila por eso.

## Guardado del dataset unificado

**Qué hacemos:** guardamos la tabla final en `data/olist_dataset_unificado.csv` y mostramos las primeras filas.

**Para qué:** la unificación es la parte "pesada" del pipeline: al guardarla en disco, los notebooks siguientes (y las corridas siguientes) cargan el CSV directamente, sin volver a cruzar 9 tablas.

> **Diferencia con el script:** `eda_completo.py` solo corre la unificación `if not CSV_UNIFICADO.exists()` (saltea si ya existe). Este notebook **la reconstruye siempre**, para que puedas ver cada forma intermedia; reescribir el mismo archivo no cambia nada (tarda pocos segundos).

In [7]:
df.to_csv(CSV_UNIFICADO, index=False)
print(f"[unificación] filas={df.shape[0]:,} columnas={df.shape[1]} -> {CSV_UNIFICADO}")
print()
print("Columnas resultantes:")
print(list(df.columns))
df.head(3)

[unificación] filas=112,650 columnas=39 -> D:\Ciencia de Datos\Proyecto Integrador\trabajo_ integrador_versionNati\trabajo_ integrador_versionNati\data\olist_dataset_unificado.csv

Columnas resultantes:
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'product_category_name_english', 'seller_zip_code_prefix', 'seller_city', 'seller_state', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'payment_value_total', 'payment_installments_max', 'payment_type_principal', 'review_score', 'review_comment_message', 'customer_lat', 'customer_lng', 'sell

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,payment_value_total,payment_installments_max,payment_type_principal,review_score,review_comment_message,customer_lat,customer_lng,seller_lat,seller_lng
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29 00:00:00,cool_stuff,58.0,598.0,4.0,650.0,28.0,9.0,14.0,cool_stuff,27277,volta redonda,SP,871766c5855e863f6eccc05f988b23cb,28013,campos dos goytacazes,RJ,72.19,2.0,credit_card,5.0,"Perfeito, produto entregue antes do combinado.",-21.762775,-41.309633,-22.496953,-44.127492
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15 00:00:00,pet_shop,56.0,239.0,2.0,30000.0,50.0,30.0,40.0,pet_shop,3471,sao paulo,SP,eb28e67c4c0b83846050ddfb8a35d051,15775,santa fe do sul,SP,259.83,3.0,credit_card,4.0,None,-20.220527,-50.903424,-23.565096,-46.518565
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.0,17.87,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05 00:00:00,moveis_decoracao,59.0,695.0,2.0,3050.0,33.0,13.0,33.0,furniture_decor,37564,borda da mata,MG,3818d81c6709e39d06b2738a8d3a2474,35661,para de minas,MG,216.87,5.0,credit_card,5.0,Chegou antes do prazo previsto e o produto sur...,-19.870305,-44.593326,-22.262584,-46.171124


**Insight:** quedó una tabla "ancha" (~112 mil filas × ~40 columnas) que trae toda la información de las 9 tablas originales en un solo lugar: identificación y estado del pedido, fechas, producto + categoría en inglés, vendedor, cliente + estado/ciudad, pago agregado, reseña y coordenadas de ambos extremos (cliente y vendedor). Esta es la materia prima de los notebooks siguientes.

**Siguiente paso:** `03_carga_y_limpieza.ipynb` — verificar la carga, limpiar y tipar los datos.